# Embedding Geometry Analysis

Geometric structure of the embedding space: isotropy, nearest neighbors,
PCA, clustering, and norm distribution.

## Why This Matters

Embedding geometry examines the structure of the model's token representation spaces. The embedding and unembedding matrices define the model's interface with language, and their geometry constrains what semantic relationships the model can represent.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.embedding_geometry_analysis import (
    embedding_isotropy, embedding_nearest_neighbors,
    embedding_pca_structure, embedding_cluster_structure,
    embedding_norm_distribution,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
print('Model ready')

## Isotropy

How uniformly are embedding directions distributed? Isotropy = 1 means
perfectly uniform; low values mean embeddings cluster in a subspace.

In [ ]:
result = embedding_isotropy(model)
print(f"Isotropy: {result['isotropy']:.4f}")
print(f"Effective dimension: {result['effective_dimension']:.2f} / {result['d_model']}")
print(f"Top eigenvalue fraction: {result['top_eigenvalue_fraction']:.4f}")
print(f"Mean embedding norm: {result['mean_norm']:.4f}")

## Nearest Neighbors

Find the closest tokens in embedding space for selected tokens.

In [ ]:
result = embedding_nearest_neighbors(model, token_ids=[1, 25, 50, 75], top_k=5)
for t in result['per_token']:
    print(f"\nToken {t['token_id']} (norm={t['norm']:.4f}):")
    for n in t['neighbors']:
        print(f"  -> Token {n['token_id']:3d}: cos={n['cosine']:.4f}")

## PCA Structure

Top principal components and their variance fractions.

In [ ]:
result = embedding_pca_structure(model, n_components=10)
print(f"Total variance: {result['total_variance']:.4f}\n")
for c in result['components']:
    bar = '█' * int(c['variance_fraction'] * 50)
    print(f"  PC{c['component']}: {c['variance_fraction']:.3f} "
          f"(cum: {c['cumulative_variance']:.3f}) {bar}")

## Cluster Structure

Are embeddings clustered or well-spread? Pairwise cosine statistics.

In [ ]:
result = embedding_cluster_structure(model, n_samples=50)
print(f"Well spread: {result['is_well_spread']}")
print(f"Mean pairwise cosine: {result['mean_pairwise_cosine']:.4f}")
print(f"Max pairwise cosine: {result['max_pairwise_cosine']:.4f}")
print(f"Fraction high similarity: {result['fraction_high_similarity']:.3f}")

## Norm Distribution

Tokens with unusual embedding norms may be special (BOS, EOS, unused).

In [ ]:
result = embedding_norm_distribution(model, top_k=5)
print(f"Mean norm: {result['mean_norm']:.4f} ± {result['std_norm']:.4f}")
print(f"Range: [{result['min_norm']:.4f}, {result['max_norm']:.4f}]\n")
print('Largest norms:')
for t in result['top_norm_tokens']:
    print(f"  Token {t['token_id']:3d}: norm={t['norm']:.4f} (z={t['z_score']:.2f})")
print('\nSmallest norms:')
for t in result['bottom_norm_tokens']:
    print(f"  Token {t['token_id']:3d}: norm={t['norm']:.4f} (z={t['z_score']:.2f})")